# Final Exam

For your final exam, you will receive a dataset with interviews / statements from executives from US companies (CEOs, CFOs, ..). From this dataset, you should answer research questions which will be presented at the 2nd of December. 

For a better understanding, we will present to you the basics how this dataset was created and how it looks like.

## Creation of the dataset

The dataset was build based on the *anncomp* table from WRDS. This table includes information about executives from US companies (CEOs, CFOs, ..) including their names, positions, companies they work for and other relevant details. You can find more information about this table [here](https://wrds-www.wharton.upenn.edu/pages/get-data/compustat-capital-iq-standard-poors/compustat/execucomp/annual-compensation/).

To create the dataset, we followed these steps:
1. **Create youtube search queries**: For each executive in the *anncomp* table, we created a youtube search query using their name and position (e.g., "John Doe CEO Tesla Interview").

2. **Scrape youtube videos**: We used the youtube search queries to scrape youtube for relevant videos. We collected the video URLs and metadata (e.g., title, description, upload date).

3. **Transcribe videos**: We used a transcription service to transcribe the audio from the videos into text. This resulted in a collection of interview transcripts/statements from the executives.

4. **Label transcripts**: For each transcript, we labeled it with the corresponding executive's information (e.g., name, position, company) from the *execucomp* table. For each executive we labelled a few transcripts/statements manually and the rest was labelled using a machine learning model trained on the manually labelled data.

5. **First cleaning step**: We performed a first cleaning step to remove videos which were duplicates. The video itself was not a duplicate because each video as a unique video_id on youtube. However, some videos were duplicates in the sense that they contained the same interview/statements from the same executive. We identified these duplicates by comparing the transcripts and removed them from the dataset.

### Column explanation

**General information**:
- exec_fullname: Full name of the executive
- coname: Name of the company the executive works for
- coname_2: Name of the company the executive works for but from different database
- video_id_anonymized: Anonymized ID of the youtube video
- speaker_id: Speaker ID of the executive in the transcript_segments
- speaker_similarity: Similarity score of the speaker in the transcript_segments to the manually labelled speaker
- fulltitles: Full titles (CEO, ..) of the executive
- video_title: Title of the youtube video
- channel_id_anonymized: Anonymized ID of the youtube channel
- video_duration: Duration of the youtube video
- view_count: View count of the youtube video
- video_description: Description of the youtube video
- ticker: Ticker of the company the executive works for
- publish_date: Publish date of the youtube video
- transcript_segments: Segments of the transcript with speaker labels
- executive_segments: Segments of the transcript spoken by the executive
- non_executive_segments: Segments of the transcript not spoken by the executive
- joined_executive_text: Joined text of the executive segments
- gvkey: GVKEY of the company the executive works for
- gsector: GICS sector of the company the executive works for

**Role indicators**: (functions are below how we computed them)
- is_CEO: Indicator if the executive is a CEO
- is_CFO: Indicator if the executive is a CFO
- is_COO: Indicator if the executive is a COO
- is_President: Indicator if the executive is a President
- is_Chair: Indicator if the executive is a Chair
- is_Director: Indicator if the executive is a Director
- is_Founder: Indicator if the executive is a Founder
- is_Tech_Science: Indicator if the executive has a technical or scientific role
- is_Legal: Indicator if the executive has a legal role
- is_EVP_SVP: Indicator if the executive is an EVP or SVP
- CEO_Duality: Indicator if the executive is a CEO and Chair
- Founder_CEO: Indicator if the executive is a Founder and CEO
- is_Exec_Chair: Indicator if the executive is an Executive Chair
- is_Interim: Indicator if the executive is an Interim executive
- Role_Category: Categorical variable indicating the role category of the executive

**Textual features**: (based on Loughran & Mcdonald dictionary)
- positive_word_count_executive: Count of positive words in the executive segments
- negative_word_count_executive: Count of negative words in the executive Segments
- strong_modal_word_count_executive: Count of strong modal words in the executive segments
- weak_modal_word_count_executive: Count of weak modal words in the executive segments
- uncertainty_word_count_executive: Count of uncertainty words in the executive segments
- constraining_word_count_executive: Count of constraining words in the executive segments
- litigious_word_count_executive: Count of litigious words in the executive segments

**Sentiment features**: (FinBERT)
- speaker_finbert_neutral: FinBERT neutral sentiment score for the executive segments
- speaker_finbert_positive: FinBERT positive sentiment score for the executive segments
- speaker_finbert_negative: FinBERT negative sentiment score for the executive segments
- speaker_lm_polarity: Language model polarity score for the executive segments
- speaker_lm_subjectivity: Language model subjectivity score for the executive segments
- context_finbert_neutral: FinBERT neutral sentiment score for the full transcript except the executive segments
- context_finbert_negative: FinBERT negative sentiment score for the full transcript except the executive segments
- context_finbert_positive: FinBERT positive sentiment score for the full transcript except the executive segments

**Audio features**: (using librosa)
- pitch_mean: Mean pitch of the executive segments
- pitch_std: Standard deviation of pitch of the executive segments
- intensity_mean: Mean intensity of the executive segments
- intensity_std: Standard deviation of intensity of the executive segments
- voiced_fraction: Voiced fraction of the executive segments
- jitter: Jitter of the executive segments
- shimmer: Shimmer of the executive segments
- hnr: Harmonics-to-noise ratio of the executive segments

**Narcissism / Readability features**:
- first_person_singular: Count of first person singular pronouns in the executive segments
- first_person_plural: Count of first person plural pronouns in the executive segments
- singular_pronoun_ratio: Ratio of first person singular to plural pronouns in the executive segments
- fog_index: Gunning fog index of the executive segments
- avg_sentence_length: Average sentence length in the executive segments
- flesch_reading_ease: Flesch reading ease score of the executive segments
- numerosity_numbers: Count of numerical words in the executive segments
- numerosity_total_words: Total word count in the executive segments
- numerosity_ratio: Ratio of numerical words to total words in the executive segments

### How did we group the roles?
The roles were grouped based on their common responsibilities and functions within a company. Here is a brief explanation of the grouping:

| Your Python Category | Literature Classification | Key Citation | Theoretical Justification |
| :--- | :--- | :--- | :--- |
| **CEO / Founder_CEO** | Decision Management / Apex | Hambrick & Mason (1984); Finkelstein (1992) | The primary architect of strategy. Founders are treated distinctively in literature due to "psychological ownership" and higher risk tolerance. |
| **CFO** | Throughput Function / Monitor | Jiang, Petroni, & Wang (2010) | Distinct from the CEO due to specific fiduciary duties, focus on efficiency, and legal liability for financial reporting (SOX). |
| **President_COO** | Throughput Function | Michel & Hambrick (1992) | Focuses on internal efficiency, implementation, and daily operations rather than external strategic vision. |
| **Tech_Science_Lead** | Output Function (R&D) | Barker & Mueller (2002); Tyler & Steensma (1998) | Executives responsible for innovation, future product pipelines, and technological capability (Output-oriented). |
| **CMO / Sales / CRO** | Output Function (Marketing) | Hambrick & Mason (1984) | Executives focused on revenue growth, market positioning, and customer acquisition (Output-oriented). |
| **Board / Chair** | Decision Control | Fama & Jensen (1983) | Roles responsible for monitoring management performance and ratifying decisions, rather than executing daily tasks. |
| **General Counsel** | Peripheral / Admin | Melone (1994) | Focuses on compliance and risk mitigation; often acts as a restraint on the strategic aggressiveness of the CEO. |

In [4]:
def add_executive_role_controls(data):
    # Ensure titles are strings and lowercase for searching
    data["clean_title"] = data["fulltitles"].astype(str).str.lower()

    # ---------------------------------------------------------
    # 1. CORE ROLE FLAGS (Not Mutually Exclusive)
    # ---------------------------------------------------------

    # The Big Three
    data["is_CEO"] = data["clean_title"].apply(
        lambda x: 1 if "ceo" in x or "chief executive" in x else 0
    )
    data["is_CFO"] = data["clean_title"].apply(
        lambda x: 1 if "cfo" in x or "chief financial" in x or "treasurer" in x else 0
    )
    data["is_COO"] = data["clean_title"].apply(
        lambda x: 1 if "coo" in x or "chief operating" in x else 0
    )

    # The President (Often combined with COO or CEO)
    data["is_President"] = data["clean_title"].apply(
        lambda x: 1 if "president" in x else 0
    )

    # Governance Roles
    data["is_Chair"] = data["clean_title"].apply(
        lambda x: 1 if "chair" in x else 0
    )  # Captures Chairman, Chairperson, Exec Chair
    data["is_Director"] = data["clean_title"].apply(
        lambda x: 1 if "director" in x or "trustee" in x or "board member" in x else 0
    )

    # Founder Status (Crucial for risk profile)
    data["is_Founder"] = data["clean_title"].apply(lambda x: 1 if "founder" in x else 0)

    # ---------------------------------------------------------
    # 2. FUNCTIONAL EXPERTISE FLAGS
    # ---------------------------------------------------------
    # Does the person have a background in Tech, Science, or Law?

    # Tech/Science (CTO, CIO, Chief Scientific Officer)
    data["is_Tech_Science"] = data["clean_title"].apply(
        lambda x: (
            1
            if any(
                t in x
                for t in [
                    "cto",
                    "cio",
                    "technology",
                    "technical",
                    "information",
                    "scientific",
                    "chief scientist",
                    "r&d",
                ]
            )
            else 0
        )
    )

    # Legal (General Counsel)
    data["is_Legal"] = data["clean_title"].apply(
        lambda x: 1 if "general counsel" in x or "chief legal" in x else 0
    )

    # Senior Management (EVP/SVP but not necessarily C-Suite)
    data["is_EVP_SVP"] = data["clean_title"].apply(
        lambda x: (
            1
            if "executive vice" in x or "senior vice" in x or "evp" in x or "svp" in x
            else 0
        )
    )

    # ---------------------------------------------------------
    # 3. INTERACTION TERMS (The "Finance" Variables)
    # ---------------------------------------------------------

    # CEO Duality: Is the CEO also the Chairman? (High Entrenchment)
    data["CEO_Duality"] = data["is_CEO"] * data["is_Chair"]

    # Founder CEO: Is the CEO also the Founder? (High Risk/High Reward)
    data["Founder_CEO"] = data["is_CEO"] * data["is_Founder"]

    # Executive Chair: A Chair who is also an Executive (Active Management vs Passive Board)
    data["is_Exec_Chair"] = data["clean_title"].apply(
        lambda x: 1 if "executive chair" in x else 0
    )

    # Interim Status: Are they just filling in? (Short term horizon)
    data["is_Interim"] = data["clean_title"].apply(lambda x: 1 if "interim" in x else 0)

    return data

## How can you join the executive information with the transcripts?

To join the transcript data with executive information from the *anncomp* table, you need to match each transcript with its corresponding executive based on their name, company, and the year of the transcript.

*Hint: if you can not join the data perfectly, you can calculate some variables (like age) based on the joins which worked.*

**This notebook and the data is permitted to be distributed and shared freely. By using this notebook, recipients agree to respect the original authorship and integrity of the content. Redistribution, copying, or sharing is not allowed. Any commercial use or modification is not allowed as well.**